# Breno Alves de Oliveira
> Resolvido utilizando o python pois não tenho acesso ao Excel no momento.
# Programação Linear 

## Metodologia
Como os modelos têm apenas duas variáveis inteiras não negativas, é possível:
1. determinar um limite superior para cada variável;
2. testar todas as combinações inteiras viáveis;
3. calcular o valor da função objetivo;
4. escolher a melhor solução.

In [1]:

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


In [6]:
# Funcoes Helpers - Apenas p/ visualização
def fmt_num(x):
    """Formata números em estilo pt-BR para exibição."""
    if isinstance(x, (np.integer, int)):
        return str(int(x))
    if isinstance(x, (np.floating, float)):
        if np.isinf(x):
            return "∞" if x > 0 else "-∞"
        if abs(x - round(x)) < 1e-10:
            return str(int(round(x)))
        s = f"{x:.10g}"
        return s.replace(".", ",")
    return str(x)

def feasible_upper_bounds(A, b):
    A = np.asarray(A, dtype=int)
    b = np.asarray(b, dtype=int)
    ub = []
    for j in range(A.shape[1]):
        candidates = [b[i] // A[i, j] for i in range(len(b)) if A[i, j] > 0]
        ub.append(min(candidates) if candidates else 30)
    return np.asarray(ub, dtype=int)

def enumerate_feasible_solutions(c, A, b):
    """Retorna todas as soluções viáveis inteiras, o valor da FO e o melhor ponto."""
    c = np.asarray(c, dtype=float)
    A = np.asarray(A, dtype=int)
    b = np.asarray(b, dtype=int)

    ub = feasible_upper_bounds(A, b)
    pts = []
    vals = []
    for x1 in range(ub[0] + 1):
        for x2 in range(ub[1] + 1):
            x = np.array([x1, x2], dtype=int)
            if np.all(A @ x <= b):
                pts.append(x)
                vals.append(float(c @ x))

    pts = np.array(pts, dtype=int)
    vals = np.array(vals, dtype=float)
    best_val = vals.max()
    opt_mask = vals == best_val
    opt_pts = pts[opt_mask]

    # escolhe uma solução ótima "canônica": menor x1 e, em caso de empate, menor x2
    order = np.lexsort((opt_pts[:, 1], opt_pts[:, 0]))
    chosen = opt_pts[order][0]

    return {
        "ub": ub,
        "feasible_points": pts,
        "objective_values": vals,
        "best_value": best_val,
        "optimal_points": opt_pts,
        "chosen_solution": chosen,
    }

def objective_interval_for_variable(c, A, b, x_star, j):
    """
    Intervalo exato do coeficiente c[j] para o qual x_star continua ótimo,
    mantendo os demais coeficientes fixos.
    """
    c = np.asarray(c, dtype=float)
    A = np.asarray(A, dtype=int)
    b = np.asarray(b, dtype=int)
    x_star = np.asarray(x_star, dtype=float)

    pts = enumerate_feasible_solutions(c, A, b)["feasible_points"]
    lower = -np.inf
    upper = np.inf

    for x in pts:
        x = np.asarray(x, dtype=float)
        if np.allclose(x, x_star):
            continue

        dx = x_star[j] - x[j]
        const_star = sum(c[k] * x_star[k] for k in range(len(c)) if k != j)
        const_x = sum(c[k] * x[k] for k in range(len(c)) if k != j)
        rhs = const_x - const_star

        if abs(dx) < 1e-12:
            # Se dx = 0, a desigualdade não depende de c[j].
            # Como x_star já é ótimo no ponto base, não restringe o intervalo.
            continue

        bound = rhs / dx
        if dx > 0:
            lower = max(lower, bound)
        else:
            upper = min(upper, bound)

    return lower, upper

def rhs_stability_interval(c, A, b, x_star, constraint_index, search_min=0, search_max=100):
    """
    Intervalo discreto (inteiro) de RHS para o qual a solução escolhida continua ótima.
    Emula o Solver reotimizando o modelo para cada RHS inteiro.
    """
    c = np.asarray(c, dtype=float)
    A = np.asarray(A, dtype=int)
    b = np.asarray(b, dtype=int)

    stable_rhs = []
    details = []
    for rhs in range(search_min, search_max + 1):
        b_new = b.copy()
        b_new[constraint_index] = rhs
        result = enumerate_feasible_solutions(c, A, b_new)
        same = np.array_equal(result["chosen_solution"], x_star)
        details.append((rhs, same, result["chosen_solution"], result["best_value"]))
        if same:
            stable_rhs.append(rhs)

    if not stable_rhs:
        return None, pd.DataFrame(details, columns=["RHS", "Estável?", "Solução ótima", "Z*"])

    return (min(stable_rhs), max(stable_rhs)), pd.DataFrame(details, columns=["RHS", "Estável?", "Solução ótima", "Z*"])

def report_model(title, c, A, b, var_names=("x1", "x2"), constr_names=None):
    c = np.asarray(c, dtype=float)
    A = np.asarray(A, dtype=int)
    b = np.asarray(b, dtype=int)
    if constr_names is None:
        constr_names = [f"Restrição {i+1}" for i in range(A.shape[0])]

    result = enumerate_feasible_solutions(c, A, b)
    x_star = result["chosen_solution"]
    z_star = result["best_value"]
    all_opt = result["optimal_points"]
    slack = b - A @ x_star
    active = [constr_names[i] for i, s in enumerate(slack) if int(s) == 0]

    # Tabela de soluções viáveis
    feas_df = pd.DataFrame(result["feasible_points"], columns=var_names)
    feas_df["Z"] = result["objective_values"].astype(int) if np.allclose(result["objective_values"], np.round(result["objective_values"])) else result["objective_values"]

    # Sensibilidade dos coeficientes da função objetivo
    coeff_rows = []
    for j, vn in enumerate(var_names):
        lo, hi = objective_interval_for_variable(c, A, b, x_star, j)
        coeff_rows.append({
            "Variável": vn,
            "Coef. base": c[j],
            "Intervalo para manter a solução escolhida": f"[{fmt_num(lo)}, {fmt_num(hi)}]",
        })
    coeff_df = pd.DataFrame(coeff_rows)

    # Sensibilidade dos RHS (discreta)
    rhs_rows = []
    rhs_tables = {}
    for i, cn in enumerate(constr_names):
        rng, detail_df = rhs_stability_interval(c, A, b, x_star, i, search_min=0, search_max=120)
        rhs_tables[cn] = detail_df
        rhs_rows.append({
            "Restrição": cn,
            "RHS base": b[i],
            "Faixa discreta em que a solução escolhida continua ótima": f"[{rng[0]}, {rng[1]}]" if rng else "sem faixa",
            "Folga no ótimo": slack[i],
        })
    rhs_df = pd.DataFrame(rhs_rows)

    # Resumo no topo
    display(Markdown(f"## {title}"))
    display(Markdown(
        f"**Solução ótima escolhida:** ({int(x_star[0])}, {int(x_star[1])})  \n"
        f"**Valor ótimo:** Z = {fmt_num(z_star)}  \n"
        f"**Todas as soluções ótimas encontradas:** {', '.join(str(tuple(map(int, p))) for p in all_opt)}"
    ))
    display(Markdown(
        f"**Folgas no ponto ótimo escolhido:** "
        + ", ".join(f"{constr_names[i]} = {fmt_num(slack[i])}" for i in range(len(slack)))
        + (f"  \n**Restrições ativas:** {', '.join(active)}" if active else "  \n**Restrições ativas:** nenhuma")
    ))

    display(Markdown("### Soluções viáveis e valor da função objetivo"))
    display(feas_df)

    display(Markdown("### Relatório de sensibilidade em relação à função objetivo"))
    display(coeff_df)

    display(Markdown("### Relatório de sensibilidade discreto em relação aos RHS"))
    display(rhs_df)

    # Conclusão automática
    active_txt = ", ".join(active) if active else "nenhuma restrição"
    conclusion = (
        f"A solução ótima do modelo é ({int(x_star[0])}, {int(x_star[1])}) com Z = {fmt_num(z_star)}. "
        f"O ponto ótimo escolhido apresenta folgas {', '.join(f'{constr_names[i]} = {fmt_num(slack[i])}' for i in range(len(slack)))} "
        f"e possui como restrições ativas {active_txt}. "
    )
    if len(all_opt) > 1:
        conclusion += "Como existem soluções ótimas múltiplas, o Solver pode retornar qualquer uma delas, dependendo da estratégia de busca. "
    conclusion += (
        "Os intervalos acima mostram a faixa em que, variando um coeficiente por vez, "
        "a solução escolhida continua ótima. Nos RHS, a análise foi feita por reotimização discreta. "
    )
    display(Markdown("### Conclusão"))
    display(Markdown(conclusion))

    return {
        "summary": {
            "x_star": x_star,
            "z_star": z_star,
            "optimal_points": all_opt,
            "slack": slack,
            "active": active,
        },
        "feasible_df": feas_df,
        "coeff_df": coeff_df,
        "rhs_df": rhs_df,
        "rhs_details": rhs_tables,
    }



# Exercício 1

**Maximizar:** $Z = 3x_1 + 2x_2$  

**Sujeito a:**  
- 2x1 + 5x2 ≤ 9
- 4x1 + 2x2 ≤ 9

Variáveis inteiras e não negativas.

In [7]:

c = np.array([3, 2], dtype=float)
A = np.array([[2, 5], [4, 2]], dtype=int)
b = np.array([9, 9], dtype=int)

report_1 = report_model(
    "Exercício 1",
    c, A, b,
    var_names=('x1', 'x2'),
    constr_names=('2x1 + 5x2 ≤ 9', '4x1 + 2x2 ≤ 9'),
)


## Exercício 1

**Solução ótima escolhida:** (2, 0)  
**Valor ótimo:** Z = 6  
**Todas as soluções ótimas encontradas:** (2, 0)

**Folgas no ponto ótimo escolhido:** 2x1 + 5x2 ≤ 9 = 5, 4x1 + 2x2 ≤ 9 = 1  
**Restrições ativas:** nenhuma

### Soluções viáveis e valor da função objetivo

,x1,x2,Z
0,0,0,0
1,0,1,2
2,1,0,3
3,1,1,5
4,2,0,6


...

,Variável,Coef. base,Intervalo para manter a solução escolhida
0,x1,3.0,"[2, ∞]"
1,x2,2.0,"[-∞, 3]"


### Relatório de sensibilidade discreto em relação aos RHS

,Restrição,RHS base,Faixa discreta em que a solução escolhida continua ótima,Folga no ótimo
0,2x1 + 5x2 ≤ 9,9,"[4, 11]",5
1,4x1 + 2x2 ≤ 9,9,"[8, 9]",1


### Conclusão

A solução ótima do modelo é (2, 0) com Z = 6. O ponto ótimo escolhido apresenta folgas 2x1 + 5x2 ≤ 9 = 5, 4x1 + 2x2 ≤ 9 = 1 e possui como restrições ativas nenhuma restrição. Os intervalos acima mostram a faixa em que, variando um coeficiente por vez, a solução escolhida continua ótima. Nos RHS, a análise foi feita por reotimização discreta. 


# Exercício 2

**Maximizar:** $Z = 2x_1 + 3x_2$  

**Sujeito a:**  
- 5x1 + 7x2 ≤ 35
- 4x1 + 9x2 ≤ 36

Variáveis inteiras e não negativas.

In [8]:

c = np.array([2, 3], dtype=float)
A = np.array([[5, 7], [4, 9]], dtype=int)
b = np.array([35, 36], dtype=int)

report_2 = report_model(
    "Exercício 2",
    c, A, b,
    var_names=('x1', 'x2'),
    constr_names=('5x1 + 7x2 ≤ 35', '4x1 + 9x2 ≤ 36'),
)


## Exercício 2

**Solução ótima escolhida:** (4, 2)  
**Valor ótimo:** Z = 14  
**Todas as soluções ótimas encontradas:** (4, 2), (7, 0)

**Folgas no ponto ótimo escolhido:** 5x1 + 7x2 ≤ 35 = 1, 4x1 + 9x2 ≤ 36 = 2  
**Restrições ativas:** nenhuma

### Soluções viáveis e valor da função objetivo

,x1,x2,Z
0,0,0,0
1,0,1,3
2,0,2,6
3,0,3,9
4,0,4,12
5,1,0,2
6,1,1,5
7,1,2,8
8,1,3,11
9,2,0,4


...

,Variável,Coef. base,Intervalo para manter a solução escolhida
0,x1,2.0,"[1,5, 2]"
1,x2,3.0,"[3, 4]"


### Relatório de sensibilidade discreto em relação aos RHS

,Restrição,RHS base,Faixa discreta em que a solução escolhida continua ótima,Folga no ótimo
0,5x1 + 7x2 ≤ 35,35,"[34, 36]",1
1,4x1 + 9x2 ≤ 36,36,"[34, 39]",2


### Conclusão

A solução ótima do modelo é (4, 2) com Z = 14. O ponto ótimo escolhido apresenta folgas 5x1 + 7x2 ≤ 35 = 1, 4x1 + 9x2 ≤ 36 = 2 e possui como restrições ativas nenhuma restrição. Como existem soluções ótimas múltiplas, o Solver pode retornar qualquer uma delas, dependendo da estratégia de busca. Os intervalos acima mostram a faixa em que, variando um coeficiente por vez, a solução escolhida continua ótima. Nos RHS, a análise foi feita por reotimização discreta. 


# Exercício 3

**Maximizar:** $Z = 1x_1 + 1x_2$  

**Sujeito a:**  
- 2x1 + 5x2 ≤ 16
- 6x1 + 5x2 ≤ 27

Variáveis inteiras e não negativas.

In [9]:

c = np.array([1, 1], dtype=float)
A = np.array([[2, 5], [6, 5]], dtype=int)
b = np.array([16, 27], dtype=int)

report_3 = report_model(
    "Exercício 3",
    c, A, b,
    var_names=('x1', 'x2'),
    constr_names=('2x1 + 5x2 ≤ 16', '6x1 + 5x2 ≤ 27'),
)


## Exercício 3

**Solução ótima escolhida:** (2, 2)  
**Valor ótimo:** Z = 4  
**Todas as soluções ótimas encontradas:** (2, 2), (3, 1), (4, 0)

**Folgas no ponto ótimo escolhido:** 2x1 + 5x2 ≤ 16 = 2, 6x1 + 5x2 ≤ 27 = 5  
**Restrições ativas:** nenhuma

### Soluções viáveis e valor da função objetivo

,x1,x2,Z
0,0,0,0
1,0,1,1
2,0,2,2
3,0,3,3
4,1,0,1
5,1,1,2
6,1,2,3
7,2,0,2
8,2,1,3
9,2,2,4


...

,Variável,Coef. base,Intervalo para manter a solução escolhida
0,x1,1.0,"[0,5, 1]"
1,x2,1.0,"[1, 2]"


### Relatório de sensibilidade discreto em relação aos RHS

,Restrição,RHS base,Faixa discreta em que a solução escolhida continua ótima,Folga no ótimo
0,2x1 + 5x2 ≤ 16,16,"[14, 16]",2
1,6x1 + 5x2 ≤ 27,27,"[22, 27]",5


### Conclusão

A solução ótima do modelo é (2, 2) com Z = 4. O ponto ótimo escolhido apresenta folgas 2x1 + 5x2 ≤ 16 = 2, 6x1 + 5x2 ≤ 27 = 5 e possui como restrições ativas nenhuma restrição. Como existem soluções ótimas múltiplas, o Solver pode retornar qualquer uma delas, dependendo da estratégia de busca. Os intervalos acima mostram a faixa em que, variando um coeficiente por vez, a solução escolhida continua ótima. Nos RHS, a análise foi feita por reotimização discreta. 


# Exercício 4

**Maximizar:** $Z = 5x_1 + 7x_2$  

**Sujeito a:**  
- 2x1 + x2 ≤ 13
- 5x1 + 9x2 ≤ 41

Variáveis inteiras e não negativas.

In [10]:

c = np.array([5, 7], dtype=float)
A = np.array([[2, 1], [5, 9]], dtype=int)
b = np.array([13, 41], dtype=int)

report_4 = report_model(
    "Exercício 4",
    c, A, b,
    var_names=('x1', 'x2'),
    constr_names=('2x1 + x2 ≤ 13', '5x1 + 9x2 ≤ 41'),
)


## Exercício 4

**Solução ótima escolhida:** (6, 1)  
**Valor ótimo:** Z = 37  
**Todas as soluções ótimas encontradas:** (6, 1)

**Folgas no ponto ótimo escolhido:** 2x1 + x2 ≤ 13 = 0, 5x1 + 9x2 ≤ 41 = 2  
**Restrições ativas:** 2x1 + x2 ≤ 13

### Soluções viáveis e valor da função objetivo

,x1,x2,Z
0,0,0,0
1,0,1,7
2,0,2,14
3,0,3,21
4,0,4,28
5,1,0,5
6,1,1,12
7,1,2,19
8,1,3,26
9,1,4,33


...

,Variável,Coef. base,Intervalo para manter a solução escolhida
0,x1,5.0,"[4,2, ∞]"
1,x2,7.0,"[0, 8,333333333]"


### Relatório de sensibilidade discreto em relação aos RHS

,Restrição,RHS base,Faixa discreta em que a solução escolhida continua ótima,Folga no ótimo
0,2x1 + x2 ≤ 13,13,"[13, 15]",0
1,5x1 + 9x2 ≤ 41,41,"[39, 42]",2


### Conclusão

A solução ótima do modelo é (6, 1) com Z = 37. O ponto ótimo escolhido apresenta folgas 2x1 + x2 ≤ 13 = 0, 5x1 + 9x2 ≤ 41 = 2 e possui como restrições ativas 2x1 + x2 ≤ 13. Os intervalos acima mostram a faixa em que, variando um coeficiente por vez, a solução escolhida continua ótima. Nos RHS, a análise foi feita por reotimização discreta. 